In [10]:
import requests
from bs4 import BeautifulSoup

import regex as re
from datetime import datetime
import time
import random

In [11]:
# Extraer todas las URLs de noticias
def extraer_urls(periodico, seccion):
    url_valida = re.compile(rf"https://{periodico}/{seccion}/[^/]+\.html$")
    urls = []

    url = f"https://{periodico}/{seccion}/"
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    for enlace in soup.find_all("a"):
        href = enlace.get("href")
        if url_valida.match(href) and (href not in urls):
            urls.append(href)
            
    return urls

In [12]:
def extraer_noticia(url, seccion, periodico):
    
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers)
    soup = BeautifulSoup(resp.text, "html.parser")

    # Título
    titulo = soup.find("h1")
    titulo = titulo.get_text(strip=True) if titulo else None

    # Subtítulo
    subtitulo = soup.find("h2")
    subtitulo = subtitulo.get_text(strip=True) if subtitulo else None

    # Texto del artículo
    parrafos = soup.find_all("p", class_="article-text")
    texto = "\n".join(p.get_text(strip=True) for p in parrafos)
    fecha_actual = datetime.now().strftime("%d-%m-%Y")

    noticia = {
            "Link": url,
            "Periódico": periodico,
            "Fecha": fecha_actual,
            "Título": titulo,
            "Subtítulo": subtitulo if subtitulo else None,
            "Categoría": seccion,
            "Contenido": texto
        }
    return noticia


In [13]:
def scrappeo_seccion(periodico, seccion):
    urls = extraer_urls(periodico=periodico, seccion=seccion)
    noticias = []
    i=1
    for url in urls:
        noticias.append(extraer_noticia(url, seccion=seccion, periodico=periodico))
        time.sleep(random.uniform(5, 10))
        print(f"Noticia {i} de {len(urls)}")
        i+=1
    return noticias

url_base = "https:/"
periodico = "www.eldiario.es"
seccion = "internacional" 

noticias = scrappeo_seccion(periodico="www.eldiario.es", seccion="internacional")

Noticia 1 de 20
Noticia 2 de 20
Noticia 3 de 20
Noticia 4 de 20
Noticia 5 de 20
Noticia 6 de 20
Noticia 7 de 20
Noticia 8 de 20
Noticia 9 de 20
Noticia 10 de 20
Noticia 11 de 20
Noticia 12 de 20
Noticia 13 de 20
Noticia 14 de 20
Noticia 15 de 20
Noticia 16 de 20
Noticia 17 de 20
Noticia 18 de 20
Noticia 19 de 20
Noticia 20 de 20


In [14]:
for noticia in noticias:
    print(" ========== ========== ========= ======== =========")
    print("------------ ###### TITULO ###### ---------")
    print(noticia["Título"])
    print("/n")
    print("------------ ###### SUBTITULO ###### ---------")
    print(noticia["Subtítulo"])
    print("/n")
    print("------------ ###### CONTENIDO ###### ---------")
    print(noticia["Contenido"])
    print("/n")

 ========== ========== ========= ======== =========
------------ ###### TITULO ###### ---------
El ataque de la cena de corresponsales pone en la diana las políticas de seguridad de la Casa Blanca
/n
------------ ###### SUBTITULO ###### ---------
La cena de corresponsales no recibió la clasificación de máxima seguridad que habría permitido movilizar todos los recursos federales disponibles, a pesar de la presencia de decenas de altos cargos, incluido la mayoría del Gabinete de Trump
/n
------------ ###### CONTENIDO ###### ---------
La Casa Blanca insiste en que la seguridad no falló el sábado pasado. Pero nada más ocurrir el ataque en el hotel de la cena de corresponsales, en la que estaba presente el presidente de EEUU, el propio Donald Trump publicó en sus redes sociales que el incidente demostraba la necesidad del salón de gala que está construyendo en la Casa Blanca sin supervisión alguna.
“Este suceso nunca habría ocurrido si se hubiera contado con el salón de máxima seguridad mil

In [15]:
# GUARDADO DE LAS NOTICIAS EN UN JSON
import json
import os

def guardar_noticias(noticias, archivo_json="./noticias_ElDiario.json"):
    
    # Si el archivo existe, cargar su contenido
    if os.path.exists(archivo_json):
        with open(archivo_json, "r", encoding="utf-8") as f:
            datos_existentes = json.load(f)
    else:
        datos_existentes = []

    # Añadir las nuevas noticias
    datos_existentes.extend(noticias)

    # Guardar todo de nuevo
    with open(archivo_json, "w", encoding="utf-8") as f:
        json.dump(datos_existentes, f, ensure_ascii=False, indent=4)

guardar_noticias(noticias=noticias)


In [16]:
with open("./noticias_ElDiario.json", "r", encoding="utf-8") as f:
            datos_existentes = json.load(f)


In [17]:

datos_existentes[2]

{'Link': 'https://www.eldiario.es/internacional/endurecimiento-bloqueo-trump-cuba-dispara-mortalidad-infantil-isla_1_13171531.html',
 'Periódico': 'www.eldiario.es',
 'Fecha': '28-04-2026',
 'Título': 'El endurecimiento del bloqueo de Trump sobre Cuba dispara la mortalidad infantil en la isla',
 'Subtítulo': 'Un informe del Center for Economic and Policy Research (CEPR) concluye que el endurecimiento de las sanciones estadounidenses contra Cuba desde 2017 y hasta 2025 es la causa principal del aumento de la mortalidad infantil en el país en un 148%, afectando a unos 1.800 bebés',
 'Categoría': 'internacional',
 'Contenido': 'El endurecimiento de las sanciones y el bloqueo contra Cuba por parte de EEUU desde 2017, durante la primera presidencia de Donald Trump, es clave en el gran aumento de la mortalidad infantil en la isla.Así lo constata un nuevo informedel Center for Economic and Policy Research (CEPR), elaborado por los investigadores\xa0Alex Main,\xa0Joe Sammut,\xa0Mark Weisbrot\x